In [3]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/03_Dataset.zip")
EXTRACT_DIR = Path("/content")

assert ZIP_PATH.exists(), "Upload 03_Dataset.zip first"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

DATA_DIR = Path("/content/03_Dataset")

assert (DATA_DIR / "metadata.csv").exists()
assert (DATA_DIR / "provenance.csv").exists()

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [4]:
import pandas as pd

metadata = pd.read_csv(DATA_DIR / "metadata.csv")
provenance = pd.read_csv(DATA_DIR / "provenance.csv")

print("Dataset shape:", metadata.shape)
print("\nColumns:")
print(metadata.columns.tolist())

print("\nMissing values:")
print(metadata.isna().sum())

print("\nDuplicate rows:", metadata.duplicated().sum())

print("\nCrop distribution:")
print(metadata["crop"].value_counts())

print("\nRice class distribution:")
rice = metadata[metadata["crop"].str.lower() == "rice"].copy()
print(rice["class_label"].value_counts())

print("\nRecommended split distribution:")
print(rice["split_recommended"].value_counts())

print("\nProvenance:")
print(provenance.to_string(index=False))

Dataset shape: (7920, 14)

Columns:
['file_path', 'unix_epoch', 'year', 'date', 'hour', 'crop', 'class_label', 'class_id', 'growth_stage', 'temperature_C', 'humidity_pct', 'pressure_hpa', 'field_id', 'split_recommended']

Missing values:
file_path            0
unix_epoch           0
year                 0
date                 0
hour                 0
crop                 0
class_label          0
class_id             0
growth_stage         0
temperature_C        0
humidity_pct         0
pressure_hpa         0
field_id             0
split_recommended    0
dtype: int64

Duplicate rows: 0

Crop distribution:
crop
Rice     5280
Jowar    2640
Name: count, dtype: int64

Rice class distribution:
class_label
Healthy          1920
Sheath_Blight    1680
Stem_Borer       1680
Name: count, dtype: int64

Recommended split distribution:
split_recommended
train    2640
test     1728
val       912
Name: count, dtype: int64

Provenance:
                                                         dataset_ve

In [5]:
import numpy as np

records = []
missing_files = []

for _, row in rice.iterrows():
    file_path = Path("/content") / row["file_path"]

    if not file_path.exists():
        missing_files.append(str(file_path))
        continue

    sensor = pd.read_csv(file_path)

    gas_cols = [
        c for c in sensor.columns
        if "cycle_" in c.lower()
    ]

    values = sensor[gas_cols].to_numpy(dtype=float)
    valid = values[np.isfinite(values)]

    records.append({
        "class_label": row["class_label"],
        "file_path": row["file_path"],
        "readings": len(valid),
        "saturation_fraction": (
            np.mean(valid >= 320000) if len(valid) else np.nan
        ),
        "median_resistance": (
            np.median(valid) if len(valid) else np.nan
        )
    })

audit = pd.DataFrame(records)

print("Rice files audited:", len(audit))
print("Missing files:", len(missing_files))

print("\nSaturation by class:")
print(
    audit.groupby("class_label")["saturation_fraction"]
    .agg(["count", "mean", "median"])
)

print("\nMedian resistance by class:")
print(
    audit.groupby("class_label")["median_resistance"]
    .agg(["mean", "median", "std"])
)

audit.to_csv(
    "/content/biosniff_rice_sensor_audit.csv",
    index=False
)

print("\nSaved biosniff_rice_sensor_audit.csv")

Rice files audited: 5280
Missing files: 0

Saturation by class:
               count      mean  median
class_label                           
Healthy         1920  0.898292    0.90
Sheath_Blight   1680  0.245095    0.22
Stem_Borer      1680  0.518488    0.52

Median resistance by class:
                        mean       median           std
class_label                                            
Healthy        320000.000000  320000.0000      0.000000
Sheath_Blight  288008.298211  286510.2975  14607.842954
Stem_Borer     316458.829863  320000.0000   5403.776449

Saved biosniff_rice_sensor_audit.csv


In [6]:
from google.colab import files

files.download("/content/biosniff_rice_sensor_audit.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>